In [ ]:
!pip install unsloth

# Model

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

# Prompt Format

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an IELTS Writing Task 2 examiner. 
Evaluate the essay focusing ONLY on the criterion **Lexical Resource (LR)**. 
Lexical Resource means how appropriately, accurately, and flexibly the writer uses vocabulary; whether a range of vocabulary is used to convey precise meaning; and whether there are errors in word choice, spelling, or collocation that affect clarity or naturalness. 
Return the result in strict JSON format with two fields:
- "score": the band score for Lexical Resource, rounded to the nearest 0.5
- "comment": an explanation justifying the score.

### Input:
Essay prompt: {}
Essay: {}

### Response:
{}"""

# 1 Preprocess

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/kaggle/input/llm-final-project/train_final.csv")

In [ ]:
print(df.describe())

In [ ]:
import matplotlib.pyplot as plt

print(df['LR_Band'].unique())
print(df['LR_Band'].dtype)

# Vẽ phân bố
df['LR_Band'].value_counts().sort_index().plot(kind="bar", edgecolor="black")

In [ ]:
first_row = df.iloc[0]
print(first_row["prompt"])
print(first_row["essay"])

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        first_row["prompt"], # prompt
        first_row["essay"], # essay
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

# 2 Fine-tune

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
import json

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def format_to_json(score, comment):
    return json.dumps({"score": score, "comment": comment})

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays       = examples["essay"]
    scores      = examples["LR_Band"]
    comments    = examples["LR_Comment"]
    texts = []
    for prompt, essay, score, comment in zip(prompts, essays, scores, comments):
        output_json = format_to_json(score, comment)
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(prompt, essay, output_json) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [ ]:
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
print(dataset[0]["text"])  # in thử dòng đầu

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for full training run.
        #max_steps = 8,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/",
        report_to = "none", # Use this for WandB etc,
    ),
)

In [ ]:
train_from_checkpoint = True
checkpoint_path = "/kaggle/input/llama-8b-lexical-resource-3epochs/tensorrtllm/default/1/checkpoint-1000"

In [ ]:
if train_from_checkpoint:
    # Resume từ checkpoint
    trainer.train(resume_from_checkpoint = checkpoint_path)
else:
    trainer_stats = trainer.train()

In [ ]:
model_name = "Llama_8b_Lexical_Resource_3epochs_model"

In [ ]:
output_dir = f"/kaggle/working/{model_name}"

model.save_pretrained(output_dir)  # Local saving
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
import zipfile
import os

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, arcname)

zip_folder(output_dir, f"{model_name}.zip")
print(f"Zip Model saved to {model_name}.zip")